# 05 - Crime Hotspot Discovery

Three complementary spatial views:

* **Grid hotspots** — bucket incidents into ~250 m × 250 m cells
* **DBSCAN density clusters** — geographic clusters with haversine metric
* **K-Means area archetypes** — cluster LAPD areas by their crime profile

In [1]:
# bootstrap: make src importable
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, data_loader, hotspot
clean = data_loader.load_clean()

02:08:13 | INFO    | src.data_loader | [start] load clean crimes_clean.parquet


02:08:14 | INFO    | src.data_loader | [done ] load clean crimes_clean.parquet in 0.40s


## Grid-based top cells

In [2]:
grid = hotspot.grid_hotspots(clean)
grid.head(15)

,lat,lon,crimes,violent,violent_share
4316,34.044998,-118.250000,2252,493,0.218917
1222,33.945000,-118.399994,1943,98,0.050437
4450,34.047501,-118.262497,1862,195,0.104726
4318,34.044998,-118.244995,1849,675,0.365062
4861,34.055000,-118.237495,1759,247,0.140421
5072,34.059998,-118.274994,1655,531,0.320846
4452,34.047501,-118.257500,1631,179,0.109749
4454,34.047501,-118.252495,1581,299,0.189121
6844,34.102501,-118.332497,1518,336,0.221344
6842,34.102501,-118.337494,1472,297,0.201766


## DBSCAN clusters

In [3]:
db = hotspot.dbscan_hotspots(clean, sample_size=40_000, eps_km=0.4, min_samples=80)
summary = hotspot.cluster_summary(db).head(15)
summary

02:08:14 | INFO    | src.hotspot | [start] DBSCAN n=40,000 eps=0.4km


02:08:15 | INFO    | src.hotspot | [done ] DBSCAN n=40,000 eps=0.4km in 0.36s


02:08:15 | INFO    | src.hotspot | DBSCAN found 26 hotspot clusters, 73.5% noise


,cluster,size,lat,lon,violent_share,top_crime
0,0,4928,34.053238,-118.268723,0.187297,BATTERY - SIMPLE ASSAULT
7,7,1443,34.100471,-118.327477,0.178794,BATTERY - SIMPLE ASSAULT
3,3,961,33.972607,-118.283844,0.298647,VEHICLE - STOLEN
9,9,330,34.165874,-118.374054,0.133333,BATTERY - SIMPLE ASSAULT
12,12,260,34.027287,-118.286362,0.076923,THEFT PLAIN - PETTY ($950 & UNDER)
11,11,229,33.990772,-118.474464,0.279476,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT"
1,1,217,34.015022,-118.347839,0.198157,THEFT OF IDENTITY
13,13,202,34.074970,-118.375435,0.094059,SHOPLIFTING - PETTY THEFT ($950 & UNDER)
14,14,200,34.042774,-118.459267,0.050000,SHOPLIFTING - PETTY THEFT ($950 & UNDER)
4,4,171,33.944500,-118.401512,0.058480,"THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LI..."


## Area archetypes via K-Means

In [4]:
daily = pd.read_parquet(config.DAILY_AREA_PARQUET)
arch = hotspot.kmeans_area_archetypes(daily, k=4)
arch.sort_values('mean_crimes', ascending=False)

,area_id,area_name,mean_crimes,std_crimes,violent,arrest,archetype
0,1,Central,39.546121,12.594964,7.120342,2.522908,1
11,12,77th Street,36.480757,10.976488,10.292608,3.270617,1
13,14,Pacific,34.081857,8.269048,3.623091,2.278558,2
2,3,Southwest,33.065974,9.985958,6.649359,2.956628,2
5,6,Hollywood,30.565058,8.792025,5.226023,2.492364,2
14,15,N Hollywood,29.661576,7.526583,3.693952,3.599878,0
17,18,Southeast,29.399511,10.102008,8.064142,1.985339,2
19,20,Olympic,29.224191,8.415651,5.097129,2.381796,2
12,13,Newton,28.787416,9.465741,5.977398,2.237630,2
6,7,Wilshire,27.937691,10.899565,3.726940,2.075748,2


## Folium interactive heatmap

This writes an HTML file under `reports/figures/` that you can open in a browser. (Re-run if needed.)

In [5]:
out = hotspot.folium_heatmap(clean, sample=80_000)
print('open:', out)

02:08:17 | INFO    | src.hotspot | folium heatmap -> C:\Users\cemil\Desktop\dataminingproject-01\reports\figures\hotspot_heatmap.html


open: C:\Users\cemil\Desktop\dataminingproject-01\reports\figures\hotspot_heatmap.html
